In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#universal schema

In [ ]:
{
    "id": str,
    "text": str,
    "source": str,          # csv / txt / parquet
    # "metadata": dict,       # original columns stored here
    "media_type": str,      # text / image / audio / video (future)
    # "file_path": str        # optional provenance
}

{'id': str, 'text': str, 'source': str, 'metadata': dict, 'media_type': str}

#Load each dataset separately and normalize

 PREPROCESSING CSV FILE

In [ ]:
# CSV FILE

import pandas as pd

df_csv = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/2.1_xml_texts.csv")

df_csv["section_title"] = df_csv["section_title"].fillna("Untitled Section")
df_csv["section_text"] = df_csv["section_text"].fillna("")

df_csv["id"] = "csv_" + df_csv.index.astype(str)

df_csv_norm = pd.DataFrame({
    "id": df_csv["id"],
    "text": (df_csv["section_title"] + "\n\n" + df_csv["section_text"]).str.strip(),
    "source": "csv",
    # "metadata": df_csv[["section_title"]].to_dict(orient="records"),
    "media_type": "text"
})

In [ ]:
df_csv_norm.to_csv('/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/2.2df_CSV_norm.csv', index=False)

In [ ]:
len(df_csv_norm)

1629018

In [ ]:
df_csv_norm.head()

,id,text,source,media_type
0,csv_0,Untitled Section\n\nPublished by the Office of...,csv,text
1,csv_1,Legal Status and Use of Seals and Logos\n\nThe...,csv,text
2,csv_2,Legal Status and Use of Seals and Logos\n\nIt ...,csv,text
3,csv_3,Use of ISBN Prefix\n\nThis is the Official U.S...,csv,text
4,csv_4,Use of ISBN Prefix\n\nU . S . G O V E R N M E ...,csv,text


# PREPROCESSING .TXT FILE CREATED THROUGH PDFs to CSV

In [ ]:
# with open("/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/1.1 merged_pdf_text.txt", "r", encoding="utf-8", errors="ignore") as f:
#     for i in range(50):
#         print(f.readline())

In [ ]:
!mkdir -p /content/data
!cp -r "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/3.1 processed_chunks/" /content/data/

In [ ]:
import re
import pandas as pd

In [ ]:
with open(
    "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/1.1 merged_pdf_text.txt",
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:
    raw_text = f.read()

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return text

    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    # remove standalone Sec.
    text = re.sub(r"^\s*Sec\.?\s*$", "", text)

    # remove leading dash artifacts (IMPORTANT for your case)
    text = re.sub(r"^—\s*", "", text)

    # clean multiple spaces again
    text = re.sub(r"\s+", " ", text)

    text = text.replace("§", "Section ")

    return text.strip()

In [ ]:
cleaned = clean_text(raw_text)

In [ ]:
def split_sections(text):
    # safer pattern (ONLY strong boundaries)
    pattern = r"(TITLE\s+\d+—|CHAPTER\s+\d+|CHAP\.)"

    parts = re.split(pattern, text)

    chunks = []
    buffer = ""

    for p in parts:
        if not p:
            continue

        if len(buffer) + len(p) > 2000:
            chunks.append(buffer.strip())
            buffer = p
        else:
            buffer += " " + p

    if buffer.strip():
        chunks.append(buffer.strip())

    return chunks

chunks = split_sections(cleaned)

In [ ]:
df_txt_norm = pd.DataFrame({
    "id": ["txt_" + str(i) for i in range(len(chunks))],
    "text": chunks,
    "source": "txt",
    "media_type": "text"
})

In [ ]:
df_txt_norm.to_csv('/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/1.2df_txt_norm.csv', index=False)

In [ ]:
len(df_txt_norm)

5797

In [ ]:
df_txt_norm.tail()

,id,text,source,media_type
5792,txt_5792,—PRESERVATION OF HISTORICAL AND ARCHEOLOGICAL ...,txt,text
5793,txt_5793,CHAPTER 3201,txt,text
5794,txt_5794,—POLICY AND ADMINISTRATIVE PROVISIONS Section ...,txt,text
5795,txt_5795,CHAPTER 3203,txt,text
5796,txt_5796,"—MONUMENTS, RUINS, SITES, AND OBJECTS OF ANTIQ...",txt,text


# chunks to csv

In [2]:
import pandas as pd
import glob
import re
import gc
import os

In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return text

    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    text = text.replace("\n", " ").replace("\r", " ")

    # remove standalone Sec.
    text = re.sub(r"^\s*Sec\.?\s*$", "", text)

    # remove leading dash artifacts (IMPORTANT for your case)
    text = re.sub(r"^—\s*", "", text)

    # clean multiple spaces again
    text = re.sub(r"\s+", " ", text)

    text = text.replace("§", "Section ")

    return text.strip()

In [4]:
files = glob.glob("/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/3.1 processed_chunks/*.parquet")
print("Total files:", len(files))

Total files: 461


In [5]:
def normalize_parquet(df, file_id):
    # ensure text column exists (adjust if needed)
    if "text" not in df.columns:
        raise ValueError(f"No 'text' column in {file_id}")

    return pd.DataFrame({
        "id": file_id + "_" + df.index.astype(str),
        "text": df["text"].astype(str),
        "source": "parquet",
        "media_type": "text"
    })

In [ ]:
# import pandas as pd
# import gc
# import os

# output_file = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/textdata/3.2final_parquet_dataset.csv"

# # remove old file if exists
# if os.path.exists(output_file):
#     os.remove(output_file)

# # track header once (FASTER + safer than repeated os checks)
# header_written = False

# for i, file in enumerate(files):

#     try:
#         df = pd.read_parquet(file)

#         # skip if no text column
#         if "text" not in df.columns:
#             del df
#             gc.collect()
#             continue

#         # normalize
#         df = pd.DataFrame({
#             "id": f"pq_{i}_" + df.index.astype(str),
#             "text": df["text"].astype(str),
#             "source": "parquet",
#             "media_type": "text"
#         })

#         # clean
#         df["text"] = df["text"].apply(clean_text)

#         # remove empty rows
#         df = df[df["text"].str.len() > 0]

#         # write to SINGLE CSV (append mode)
#         df.to_csv(
#             output_file,
#             mode="a",
#             header=not header_written,
#             index=False,
#             encoding="utf-8"
#         )

#         header_written = True

#         print(f"Saved {i+1}/{len(files)}")

#         # memory cleanup
#         del df
#         gc.collect()

#     except Exception as e:
#         print(f"Error in {file}: {e}")
#         gc.collect()

In [13]:
import pandas as pd
import glob
import os
import gc

# ONLY READ LOCAL FILE LIST (Drive only used for input listing)
files = sorted(glob.glob(
    "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/3.1 processed_chunks/*.parquet"
))

# LOCAL OUTPUT ONLY (NO DRIVE USED DURING PROCESSING)
local_output = "/content/22_40final.csv"

start_index = 22

header_written = os.path.exists(local_output)


def clean_series(s):
    s = s.astype(str)
    s = s.str.replace("\n", " ", regex=False)
    s = s.str.replace("\r", " ", regex=False)
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.replace(r"^—\s*", "", regex=True)
    s = s.str.replace("§", "Section ")
    return s.str.strip()


# =========================
# LOCAL PROCESSING ONLY
# =========================
for i, file in enumerate(files[start_index:], start=start_index):

    try:
        df = pd.read_parquet(file)

        if "text" not in df.columns:
            continue

        df = pd.DataFrame({
            "id": f"pq_{i}_" + df.index.astype(str),
            "text": df["text"].astype(str),
            "source": "parquet",
            "media_type": "text"
        })

        df["text"] = clean_series(df["text"])
        df = df[df["text"].str.len() > 0]

        # WRITE ONLY TO LOCAL DISK
        df.to_csv(
            local_output,
            mode="a",
            header=not header_written,
            index=False,
            encoding="utf-8"
        )

        header_written = True

        print(f"✅ Saved chunk {i} | rows: {len(df)}")

        del df
        gc.collect()

    except Exception as e:
        print(f"❌ Crash at chunk {i}: {e}")
        break


print("🏁 LOCAL CSV BUILD COMPLETE (NO DRIVE INVOLVED)")

✅ Saved chunk 22 | rows: 19320
✅ Saved chunk 23 | rows: 19096
✅ Saved chunk 24 | rows: 19957
✅ Saved chunk 25 | rows: 19876
✅ Saved chunk 26 | rows: 19882
✅ Saved chunk 27 | rows: 19478
✅ Saved chunk 28 | rows: 19081
✅ Saved chunk 29 | rows: 19415
✅ Saved chunk 30 | rows: 19698
✅ Saved chunk 31 | rows: 19513
✅ Saved chunk 32 | rows: 19337
✅ Saved chunk 33 | rows: 18875
✅ Saved chunk 34 | rows: 18877
✅ Saved chunk 35 | rows: 19992
✅ Saved chunk 36 | rows: 19530
✅ Saved chunk 37 | rows: 18699
✅ Saved chunk 38 | rows: 19239
✅ Saved chunk 39 | rows: 19679
✅ Saved chunk 40 | rows: 19121
✅ Saved chunk 41 | rows: 18969
✅ Saved chunk 42 | rows: 19396
✅ Saved chunk 43 | rows: 19637
✅ Saved chunk 44 | rows: 19155
✅ Saved chunk 45 | rows: 18569
✅ Saved chunk 46 | rows: 19898
✅ Saved chunk 47 | rows: 18180


KeyboardInterrupt: 

In [ ]:
# import shutil
# import os

# # Local completed file (your final CSV in Colab VM)
# local_file = "/content/3.6final.csv"

# # Destination in Google Drive
# drive_file = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/4 final_parquet_dataset.csv"

# # Safety check
# if not os.path.exists(local_file):
#     raise FileNotFoundError("❌ Local CSV file not found. Processing not completed.")

# # Copy ONLY AFTER FULL COMPLETION
# try:
#     shutil.copy(local_file, drive_file)
#     print("🚀 SUCCESS: Full CSV uploaded to Google Drive")
# except Exception as e:
#     print("❌ Upload failed:", e)